# MVP top-10 + R1 кандидат — v24 / Run All

Одна фиксированная система: ranking MVP, кандидат/отказ full-train R1 equal3 v18. **Без обучения и подбора порога.**

**Restart Kernel → Run All** в прежнем `.venv`. CPU, без интернета и ограничения времени. Сначала проверяем сохранённые признаки, затем заново обрабатываем исходные validation-изображения.

Рабочее приложение и MVP не меняются. Это development-проверка на уже наблюдавшихся данных, не независимый тест. Ожидается сохранение ranking, а не повышение mAP.


In [1]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/dual_role_experiment.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Markdown, display
from training import dual_role_experiment as experiment
SOURCE_RUN = 'comparison_v1'
RUN_NAME = 'dual_role_v1'
ALLOW_OUTER_EVALUATION = True  # Согласованная development-проверка одной фиксированной системы.
print('Repository:', REPO, '\nKernel:', sys.executable)
print('Run:', RUN_NAME, '| CPU | обучение: нет | свежий проход по validation: да | лимита времени нет')


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK 
Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Run: dual_role_v1 | CPU | обучение: нет | свежий проход по validation: да | лимита времени нет


## 1. Проверки компонентов
Синтетические данные и маленькие ONNX: независимость ролей/query, реальные векторы, экспорт и отказ, воспроизводимость. Реальные weights не меняются.


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_dual_role_inference.py', 'tests/test_dual_role_experiment.py', 'tests/test_final_model_comparison.py'], check=True)


..................................                                       [100%]
34 passed in 3.84s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_dual_role_inference.py', 'tests/test_dual_role_experiment.py', 'tests/test_final_model_comparison.py'], returncode=0)

## 2. Preflight и фиксация профиля
Проверяем завершённый v22 и неизменность исходных данных/весов. При несовпадении остановиться, не обходить проверку. Порог читается из сохранённого R1 bundle.


In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Results:', context['output'])
display(context['manifest']['dual_role_plan']['profile'])


Preflight OK. Frozen threshold=0.534365177; no fitting, no model selection.
Results: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_24_dual_role/runs/dual_role_v1


{'schema': 'dual-role-v1',
 'layout': {'dimension': 2048,
  'ranking_slice': [0, 512],
  'candidate_slice': [512, 2048],
  'normalization': 'unit blocks, no global renormalization'},
 'roles': {'ranking': 'MVP legacy k20/k3/lambda0.50',
  'candidate': 'R1 equal3 raw_top1',
  'confidence': 'maximum raw cosine of R1 block only'},
 'mvp_preprocessing': 'exif-rgb-bbox-bilinear208-imagenet-l2-v1',
 'mvp': {'path': '../../../../models/osnet_ain_x1_0_vehicle_reid_hpo_best_map.onnx',
  'sha256': '01466f503232467224774b6e3bafe6c4393b1de30b47b1bd714908a62f2006a2'},
 'r1_bundle': {'path': '../../../variant_18_retrieval_policy/runs/policy_v1/final/R1_equal3/less_graph/raw_top1/bundle.json',
  'sha256': 'f911643a916f4287b0d78941cc1c35db228bef1153a195beb9e9789f580c5fcc'},
 'threshold': 0.534365177154541,
 'calibration': {'split': 'calibration',
  'candidate_policy': 'raw_top1',
  'method': 'maximize C=0.7F1+0.3TNR; ties F1 then higher threshold',
  'protocol_sha256': '7084dc8a0b795ddf1d569b81adc1d76

## 3. Общий экспорт и новый проход по изображениям
Этапы 1–2 используют исходные кэши, этап 3 запускает четыре encoder на всех validation-изображениях, этап 4 пишет отчёт. Видны этап, прогресс изображений и время. Готовые этапы при возобновлении помечаются CACHED.


In [4]:
results = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)
print('Status:', results['status'], '| MVP unchanged:', results['protected_unchanged'])



STAGE 1/4: calibration, cached source blocks -> shared export
[START] cached_calibration | elapsed 0.00 h
[DONE] cached_calibration | 0.0 min

STAGE 2/4: validation, cached source blocks -> shared export
[START] cached_validation | elapsed 0.00 h
[DONE] cached_validation | 0.0 min

STAGE 3/4: fresh validation images, four original ONNX encoders, batch16
[START] fresh_validation | elapsed 0.00 h
Dual-role CPU: 16/1205 images
Dual-role CPU: 176/1205 images
Dual-role CPU: 336/1205 images
Dual-role CPU: 496/1205 images
Dual-role CPU: 656/1205 images
Dual-role CPU: 816/1205 images
Dual-role CPU: 976/1205 images
Dual-role CPU: 1136/1205 images
Dual-role CPU: 1205/1205 images
[DONE] fresh_validation | 2.0 min

STAGE 4/4: exact decision parity verified; report; MVP unchanged
Status: complete | MVP unchanged: True


## 4. Результат
mAP обязан совпасть с MVP; кандидаты, confidence и отказы — с full-train R1. F1 и TNR показываются отдельно. Баллы за скорость здесь не оцениваются. Приложение автоматически не переключается.


In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
print('Fresh export:', context['output'] / 'tasks/fresh_validation/export')
print('Frozen profile:', context['profile_path'])


# v24 — MVP ranking + full-train R1 candidate

Статус: **complete**.
Профиль фиксирован заранее; нет обучения, подбора порога и выбора по текущим метрикам.
Это post-hoc development-проверка на уже наблюдавшихся данных, не независимый тест.

| Этап | mAP@10 | F1 | TNR | C | 45×mAP+10×C | Принято / отказ |
|---|---:|---:|---:|---:|---:|---:|
| cached_calibration | 0.787185 | 0.813483 | 0.704918 | 0.780914 | 43.2324 | 220 / 87 |
| cached_validation | 0.814689 | 0.795455 | 0.709677 | 0.769721 | 44.3582 | 218 / 91 |
| fresh_validation | 0.814689 | 0.795455 | 0.709677 | 0.769721 | 44.3582 | 218 / 91 |

Против MVP: ΔmAP +0.000000; ΔC +0.022600; Δдвух компонентов +0.2260 балла из 55.
Рост относится к выбору кандидата, а не к ranking. Это не полный конкурсный балл.

Новый проход по изображениям: {'MVP_top10_exact': True, 'R1_candidate_confidence_refusal_exact': True, 'v22_csv_byte_parity': True, 'vector_atol': 2e-05, 'max_abs_error': 0.0, 'bit_exact_vectors': True}.
Принятый R1-кандидат отличается от top-1 MVP у 94 запросов; вне top-10 MVP у 5. Confidence относится только к кандидату R1.

## Артефакты и ограничения

- В каждом tasks/*/export: submission.csv (10 ID при любом отказе), candidates.csv, embeddings.npy, embedding_order.json.
- Настоящие float32 признаки: столбцы [0:512] MVP, [512:2048] R1. Блоки единичные, общая норма sqrt(2). Глобальная L2 не обязательна по Q&A №8.
- Сохранённый NPY воспроизводит решения через общий dual-role scorer. Референсный cosine всего вектора не равен финальному ranking: роли описаны отдельно.
- Порог и веса заморожены; выбор кандидата независим от submission (Q&A №13–14, №31). Номерная зона не обрабатывается, bbox/validation не исправляются.
- R1-реранкер не нужен для выбора raw-кандидата. Ranking использует неизменённый MVP legacy20/3/0.50.
- Cached этапы не являются новыми измерениями скорости. Fresh validation использует изображения; при возобновлении готовый этап отмечен CACHED.
- Junk/top-10: исходный evaluator не меняется. Десять записанных ID не позволяют восстановить 11-е место после удаления junk.
- Только CPU-прототип качества. Нет проверки официальной GPU-скорости, Linux, номерного сигнала или полной готовности к сдаче.
- dual_role_profile.json ссылается на веса в исследовательском репозитории: это не переносимая поставка. MVP и приложение не переключаются.
- Защищённые файлы неизменны: True; promoted=False. Время текущего вызова: 2.2 мин.


Fresh export: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_24_dual_role/runs/dual_role_v1/tasks/fresh_validation/export
Frozen profile: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_24_dual_role/runs/dual_role_v1/dual_role_profile.json
